### Teachers in general have a lot of administrations to do and one of those things is grading. Can we create a simple grade assistant to assist a Swedish teacher in grading? This exercise focuses a lot in prompt engineering and afterwards to postprocess the output using Pydantic.

a) Go into this page with examples of students answers to a particular question. Copy some example texts and paste it into files with names like student_text_1.txt, student_text_2.txt.

b) Read these data into python and tell your LLM to grade them.

In [3]:
with open('../data/Studenttexter_för_llm/student_text_1.txt', 'r', encoding='utf-8') as file:
    student_text1 = file.read()
    
with open('../data/Studenttexter_för_llm/student_text_2.txt', 'r', encoding='utf-8') as file:
    student_text2 = file.read()

In [4]:
student_text1

'Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bli\nutstöt från gruppen. Letm

In [5]:
student_text2

'Inför allas blickar\nAtt ha muntliga presentationer är något som uppleves skrämmande i dagens\nsamhälle för att vi lever i ett samhälle där man oftast inte blir tvingade till att\ngöra något. Muntliga presentationer är något man, i de flesta fall brukar bli\ntvungen att göra i skolan först. Det händer när man är i den åldern då man\ntänker väldigt mycket på vad ens vänner tycker om en. Så att behöva stå framför\nde och presentera något kan kännas skrämmande för då tror man att de som\ntittar på dig dömer dig. Vilket blir något som fastar med en även när man blir\näldre. Muntliga presentationer är inte något man gör i skolan lika ofta som att\nlämna in en uppgift så man blir aldrig riktigt van med det. Att bli van med att\ntala framför andra är ett bra sätt att bota talängslan. Desto mer man pratar inför\nen grupp människor ju mer inser man att det inte är så farligt och man kommer\nsäkert komma på några knep på att kunna hantera sina nerver om man börjar att\nprata inför andra mycket 

CONNECT TO GEMINI

In [10]:
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client()

def ask_llm(promt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=promt
    )

    return response.text


context = """
Du är ämneslärare i svenska och ska bedömma två texter och sätta betyg på dessa enligt
svensk skolas betygssystem.
"""

prompt = context + "\nHär är elevens text:\n" + student_text1
prompt2 = context + "\nHär nästa elevs text:\n" + student_text2

# response = ask_llm(prompt, context)
response = ask_llm(prompt)
response2 = ask_llm(prompt2)

In [11]:
print(response)

Absolut, här kommer en bedömning av elevens text enligt svensk skolas betygssystem.

---

**Bedömning av elevtext**

**Text: "Inför allas blickar"**

**Helhetsintryck och ämnesområde:**
Texten behandlar ämnet talrädsla, vilket är relevant och intressant. Eleven lyckas kommunicera sin egen upplevelse och sina tankar kring ämnet, samt att koppla det till en extern källa (Peter Letmark). Texten är begriplig och förmedlar en tydlig åsikt.

**Innehåll och argumentation:**
Eleven inleder texten med att beskriva talrädslan utifrån en rädsla för att bli bedömd negativt, vilket är en känd och relevant aspekt av ämnet. Den personliga reflektionen "Jag känner mig inte så säker..." förstärker detta. Det är positivt att eleven kopplar den personliga upplevelsen till en vetenskaplig förklaring genom att referera till Peter Letmark och evolutionära förklaringar som att bli utstött från gruppen eller undvika dominanskonflikter. Detta visar en ansats till att bredda analysen. Eleven relaterar även Letm

In [13]:
print(response2)

Här är en bedömning av elevens text enligt svensk skolas betygssystem.

---

**Ämne:** Svenska
**Uppgift:** Muntliga presentationer
**Elev:** [Ej namngiven]

**Betyg:** E

**Bedömning:**

Eleven har skrivit en text som behandlar ämnet muntliga presentationer och rädslan kring dessa. Texten är engagerande och utgår från en relevant problematik i dagens samhälle, vilket är positivt.

**Innehåll och argumentation:**
Texten inleds med en tydlig problemformulering om att muntliga presentationer upplevs som skrämmande, särskilt i ung ålder då sociala aspekter är framträdande. Elevens egen röst är närvarande och bidrar till en personlig touch. Argumentationen om vikten av att öva för att hantera talängslan är relevant och väl underbyggd. Införandet av Peter Letmarks artikel ger texten ett tydligt källstöd och lyfter in intressanta perspektiv kring rädslans historiska och evolutionära ursprung, samt dess konsekvenser för samhällsutvecklingen. Att koppla den individuella rädslan till ett större

c) Prompt to get an output of fields proposed_grade, motivation and improvements.

In [21]:
def updated_ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )
    return response.text


context = """
Du är ämneslärare i svenska och ska bedömma två texter och sätta betyg på dessa enligt
svensk skolas betygssystem. 

Du ska returnera 3 fält, proposed_grade, motivation och improvements.
"""

prompt = context + "\nHär är text för elev 1: " + student_text1 + "\nHär är text för elev 2: " + student_text2

# response = ask_llm(prompt, context)
response = updated_ask_llm(prompt)


In [23]:
print(response)

[
  {
    "proposed_grade": "E",
    "motivation": "Texten behandlar ämnet talrädsla utifrån en personlig ingång som sedan knyts an till externa förklaringar från Peter Letmarks artikel. Eleven visar på en grundläggande förståelse för ämnet och försöker att koppla samman sina egna upplevelser med källmaterialet. Argumentationen är dock ibland något grundläggande och saknar djupare analys, till exempel när eleven enbart konstaterar att 'åsikterna är nästan desamma' istället för att utveckla likheter eller skillnader. Språket är funktionellt men innehåller många stavfel (t.ex. 'andledningarna', 'tal rädd', 'klass rummet', 'själv förtroende'), särskrivningar, grammatiska fel ('dem' istället för 'de', perspektivskifte i 'blir du' och dubbel negation i 'utan att inte veta') samt en del klumpiga formuleringar. Dessa brister påverkar läsbarheten och textens kvalitet negativt.",
    "improvements": "Fokusera på att granska texten noggrant för stavfel, särskrivningar och grammatiska fel (t.ex. 

In [24]:
import json

data = json.loads(response)

for i, student in enumerate(data, start=1):
    print(f"Elev {i}:")
    print("  Betyg:", student["proposed_grade"])
    print("  Motivation:", student["motivation"])
    print("  Förbättringar:", student["improvements"])
    print()


Elev 1:
  Betyg: E
  Motivation: Texten behandlar ämnet talrädsla utifrån en personlig ingång som sedan knyts an till externa förklaringar från Peter Letmarks artikel. Eleven visar på en grundläggande förståelse för ämnet och försöker att koppla samman sina egna upplevelser med källmaterialet. Argumentationen är dock ibland något grundläggande och saknar djupare analys, till exempel när eleven enbart konstaterar att 'åsikterna är nästan desamma' istället för att utveckla likheter eller skillnader. Språket är funktionellt men innehåller många stavfel (t.ex. 'andledningarna', 'tal rädd', 'klass rummet', 'själv förtroende'), särskrivningar, grammatiska fel ('dem' istället för 'de', perspektivskifte i 'blir du' och dubbel negation i 'utan att inte veta') samt en del klumpiga formuleringar. Dessa brister påverkar läsbarheten och textens kvalitet negativt.
  Förbättringar: Fokusera på att granska texten noggrant för stavfel, särskrivningar och grammatiska fel (t.ex. 'anledningarna', 'talrädd

d) Now validate this with pydantic model

In [27]:
from pydantic import BaseModel

class Grade(BaseModel):
    proposed_grade: str
    motivation: str
    improvements: str
    
class GradeList(BaseModel):
    objects: list[Grade]
    
grades = GradeList.model_validate({"objects": json.loads(response)})
grades

GradeList(objects=[Grade(proposed_grade='E', motivation="Texten behandlar ämnet talrädsla utifrån en personlig ingång som sedan knyts an till externa förklaringar från Peter Letmarks artikel. Eleven visar på en grundläggande förståelse för ämnet och försöker att koppla samman sina egna upplevelser med källmaterialet. Argumentationen är dock ibland något grundläggande och saknar djupare analys, till exempel när eleven enbart konstaterar att 'åsikterna är nästan desamma' istället för att utveckla likheter eller skillnader. Språket är funktionellt men innehåller många stavfel (t.ex. 'andledningarna', 'tal rädd', 'klass rummet', 'själv förtroende'), särskrivningar, grammatiska fel ('dem' istället för 'de', perspektivskifte i 'blir du' och dubbel negation i 'utan att inte veta') samt en del klumpiga formuleringar. Dessa brister påverkar läsbarheten och textens kvalitet negativt.", improvements="Fokusera på att granska texten noggrant för stavfel, särskrivningar och grammatiska fel (t.ex. 'a

e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

In [29]:
with open("output_data/proposed_grade.txt", "w", encoding="utf-8") as f:
    for grade in grades.objects:
        f.write(grade.proposed_grade + "\n")

with open("output_data/motivation.txt", "w", encoding="utf-8") as f:
    for grade in grades.objects:
        f.write(grade.motivation + "\n\n")

with open("output_data/improvements.txt", "w", encoding="utf-8") as f:
    for grade in grades.objects:
        f.write(grade.improvements + "\n\n")

f) Go into skolverket for Svenska 1 and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Paste this into a file called criterias.txt.

g) Now repeat b)-e) but with the criterias in your prompt as well. Can you see any differences in the outputs?

## Read file

In [30]:
with open('../data/Studenttexter_för_llm/criterias.txt', 'r', encoding='utf-8') as file:
    criterias = file.read()

## Ask with criterias

In [31]:
def ask_llm_with_criterias(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )
    return response.text


context = """
Du är ämneslärare i svenska och ska bedömma två texter och sätta betyg på dessa enligt
svensk skolas betygssystem. 

Du ska returnera 3 fält, proposed_grade, motivation och improvements.
"""

prompt = criterias + context + "\nHär är text för elev 1: " + student_text1 + "\nHär är text för elev 2: " + student_text2

response = ask_llm_with_criterias(prompt)

In [34]:
grades_with_criteria = GradeList.model_validate({"objects": json.loads(response)})
print(grades_with_criteria)

objects=[Grade(proposed_grade='E', motivation="Elev 1:s text är sammanhängande och begriplig, där eleven förmedlar egna tankar och åsikter om talrädsla samt anknyter till en extern källa (Peter Letmark). Sammanfattningen av Letmarks poänger är enkel men relevant för ämnet. Texten visar en viss anpassning till syfte och mottagare. Dock är brister i språkriktigheten, såsom stavfel (t.ex. 'andledningarna', 'tal rädd', 'klass rummet', 'själv förtroende') och vissa grammatiska tveksamheter ('folket som lyssnar på', 'blir du väldigt osäker'), framträdande och påverkar helhetsintrycket negativt. Reflektionen över källan är grundläggande, snarare en instämmande än en kritisk granskning.", improvements='För att förbättra sin text bör eleven fokusera på språkriktighet genom att noggrant granska stavning, särskilt sammansatta ord. Det är även viktigt att arbeta med meningsbyggnad och pronomenanvändning för att göra språket mer flytande och formellt korrekt. Eleven kan utveckla sin källhantering g

In [35]:
grades_with_criteria.objects

[Grade(proposed_grade='E', motivation="Elev 1:s text är sammanhängande och begriplig, där eleven förmedlar egna tankar och åsikter om talrädsla samt anknyter till en extern källa (Peter Letmark). Sammanfattningen av Letmarks poänger är enkel men relevant för ämnet. Texten visar en viss anpassning till syfte och mottagare. Dock är brister i språkriktigheten, såsom stavfel (t.ex. 'andledningarna', 'tal rädd', 'klass rummet', 'själv förtroende') och vissa grammatiska tveksamheter ('folket som lyssnar på', 'blir du väldigt osäker'), framträdande och påverkar helhetsintrycket negativt. Reflektionen över källan är grundläggande, snarare en instämmande än en kritisk granskning.", improvements='För att förbättra sin text bör eleven fokusera på språkriktighet genom att noggrant granska stavning, särskilt sammansatta ord. Det är även viktigt att arbeta med meningsbyggnad och pronomenanvändning för att göra språket mer flytande och formellt korrekt. Eleven kan utveckla sin källhantering genom att

h) Can you improve the output quality by providing few shot examples?